# Evaluate **your** KV cache compression method under MBE

**Matched-Budget Evaluation (MBE)** compares KV cache compression methods at *fixed memory budgets* on a 7–8B model, and produces a reproducible **KV Compression Card** you can cite and submit.

- **Bring your own method** (Option A): paste one `compress()` function — get a card in well under an hour.
- **Or run the built-in baselines** (Option B): KIVI quant, StreamingLLM, H2O.

**Setup:** `Runtime → Change runtime type → GPU` (a free **T4 16 GB** is enough; the 7–8B model is loaded in 4-bit). Then run the cells top to bottom.

Repo: https://github.com/rohithreddybc/mbe-protocol · Dataset: https://huggingface.co/datasets/Rohithreddybc/kv-cache-compression-mbe · Leaderboard: https://huggingface.co/spaces/Rohithreddybc/kv-cache-compression-leaderboard

In [ ]:
#@title 1 · Install + check GPU
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub
import torch; print('CUDA:', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/rohithreddybc/mbe-protocol
%cd mbe-protocol/eval

## Option A — bring your own method
Implement `compress(legacy_cache, seqlen, budget, attn_importance) -> legacy_cache`.
`legacy_cache` is a tuple of `(key, value)` per layer, each shaped `[batch, kv_heads, seq, head_dim]`. Quantize, evict, merge — whatever your method does — then return the modified cache. The example below is simple attention-guided eviction; **replace its body with your method.**

In [ ]:
#@title 2A · Define your method, then run it under MBE
user_method = '''
import torch
def compress(legacy, seqlen, budget, attn_importance):
    # EXAMPLE: keep the top-`budget` tokens by attention importance (+ first 4 sink tokens).
    # Replace this body with YOUR method.
    keep = max(5, int(round(budget * seqlen)))
    sink = list(range(min(4, seqlen)))
    top = torch.topk(attn_importance, min(keep - len(sink), seqlen)).indices.tolist()
    idx = sorted(set(sink + top))
    idx = torch.tensor(idx, dtype=torch.long, device=legacy[0][0].device)
    return tuple((k.index_select(2, idx), v.index_select(2, idx)) for k, v in legacy)
'''
open('user_method.py','w').write(user_method)
import os
os.environ['MBE_MODEL']      = 'Qwen/Qwen2.5-7B-Instruct'  #@param ['Qwen/Qwen2.5-7B-Instruct','meta-llama/Llama-3.1-8B-Instruct','Qwen/Qwen2.5-3B-Instruct']
os.environ['MBE_LOAD_4BIT']  = '1'
os.environ['MBE_N']          = '20'
os.environ['MBE_USER_METHOD']= 'user_method.py'
os.environ['MBE_USER_NAME']  = 'MyMethod'   #@param {type:'string'}
!python run_seed.py

## Option B — run the built-in baselines
(Skip if you used Option A.) Produces cards for KIVI quant, StreamingLLM, and H2O.

In [ ]:
#@title 2B · Baselines
import os
os.environ.pop('MBE_USER_METHOD', None)
os.environ['MBE_MODEL']     = 'Qwen/Qwen2.5-7B-Instruct'
os.environ['MBE_LOAD_4BIT'] = '1'
os.environ['MBE_N']         = '20'
os.environ['MBE_SKIP_H2O']  = '0'   # set 1 if attention output OOMs
!python run_seed.py

In [ ]:
#@title 3 · Show your card
import json, glob
for f in sorted(glob.glob('../cards/*.json')):
    print('===', f, '==='); print(json.dumps(json.load(open(f)), indent=2))

In [ ]:
#@title 4 · Cite MBE (paste this into your paper)
bibtex = r'''@article{reddy2026mbe,
  title  = {Breaking the Memory Wall: A Survey of Key-Value (KV) Cache Compression for Efficient Large Language Model (LLM) Inference},
  author = {Reddy B. C., Rohith and others},
  journal= {Artificial Intelligence Review},
  year   = {2026},
  note   = {MBE protocol and harness: https://github.com/rohithreddybc/mbe-protocol}
}'''
print(bibtex)

In [ ]:
#@title 5 · (Optional) Publish your card to the Hub
from huggingface_hub import login, HfApi, create_repo
login()  # paste a WRITE token: https://huggingface.co/settings/tokens
import glob, os
api = HfApi(); repo = f"{api.whoami()['name']}/kv-cache-compression-mbe"
create_repo(repo, repo_type='dataset', exist_ok=True)
for f in glob.glob('../cards/*.json'):
    api.upload_file(path_or_fileobj=f, path_in_repo='cards/'+os.path.basename(f), repo_id=repo, repo_type='dataset')
print('Published:', 'https://huggingface.co/datasets/'+repo)
print('Then open a PR / issue at the MBE repo to add it to the central leaderboard.')